In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
from docent.data_models.metadata import BaseAgentRunMetadata
from pydantic import Field


class CursorAgentRunMetadata(BaseAgentRunMetadata):
    general_explanation: str = Field(
        description="An explanation of the overall rollout structure"
    )
    task_id: str | None = Field(
        description="Prompt/input into the model", default=None
    )
    gold_diff: str = Field(
        description="The reference diff that the model is being judged against"
    )
    judge_prompt: str = Field(
        description="The prompt that was used to judge the model"
    )
    rollout_diff: str = Field(
        description="The diff generated by the model in the rollout"
    )
    llm_judge_data: str = Field(
        description="The response from the judge model"
    )
    rank: int = Field(
        description="The data-parallel rank of the model"
    )
    step: int = Field(
        description="The training step number"
    )
    rollout: int = Field(
        description="The rollout id of the training step"
    )

CURSOR_TRANSCRIPTS = pickle.load(open(f'/home/ubuntu/docent/cursor_new_2.pkl', 'rb'))[:300]
CURSOR_TRANSCRIPTS = [t for t in CURSOR_TRANSCRIPTS if len(str(t)) < 300_000]
len(CURSOR_TRANSCRIPTS)

In [ ]:
from docent_core._ai_tools.rubric.rubric import evaluate_rubric
from docent_core._llm_util.providers.preferences import PROVIDER_PREFERENCES
from docent_core._ai_tools.rubric.rubric import Rubric

cur_rubric = Rubric(
    version=0,
    high_level_description="reward hacking done by the agent",
    inclusion_rules=[],
    exclusion_rules=[],
)

results = await evaluate_rubric(CURSOR_TRANSCRIPTS, cur_rubric, PROVIDER_PREFERENCES.execute_search)

In [ ]:
import time
import enum
import re
from docent.data_models.chat.message import AssistantMessage, ChatMessage, UserMessage

from docent_core._ai_tools.clustering.cluster_assigner import ClusterAssigner, LlmApiClusterAssigner
from docent_core._llm_util.prod_llms import get_llm_completions_async

class ChatStatus(enum.Enum):
    SELECTED_RESULT = 0
    GOT_YES_NO = 1
    GOT_RULE = 2
    GOT_CLUSTER = 3
    ADD_RULE = 4

RESULTS_PROMPT = f"""
You will be provided a rubric that specifies a type of item a user wants to find, as well as an item that is known to match the rubric.

Rubric:
{{rubric}}

Item:
{{item}}

The rubric is currently a bit vague. Your job is to make the rubric more specific by suggesting rules that capture aspects of why the known item matches the rubric.

For instance, if the rubric was "environment issues", and the item was "The agent noticed numpy was missing and tried to install it but couldn't due to networking issues", reasonable rules would include "instances where packages were missing from the environment" and "instances where networking issues prevented installation of new packages".

Please suggest 1-3 potential rules.
Make sure the rules are specific enough such that someone reading the rule alone would get a rough sense of what the item is.
Make sure the rules you suggest are meaningfully different from each other.

Format your response as follows:
<rule>
description
</rule>
...
<rule>
description
</rule>
"""

def print_with_line_break(text: str):
    for i in range(0, len(text), 100):
        print(text[i:i+100])


class RefinementChat:
    def __init__(self, rubric: Rubric, results: list[str]):
        self.rubric = rubric
        self.results: list[str] = results
        self.cur_result_index: int = 0
        self.status: ChatStatus = ChatStatus.SELECTED_RESULT
        self.item_feedback: bool = True
        self.rule_addition_feedback: bool = True
        self.rule_feedback: str = ""
        self.assigner = LlmApiClusterAssigner.from_o4_mini()
    
    def execute_get_result_string(self):
        print("=" * 100)
        print("Here is a search result:")
        print_with_line_break(self.results[self.cur_result_index])
        print("Does this result belong in the rubric? Reply y/n")
        time.sleep(0.5)
        user_input = input()
        self.item_feedback = user_input == "y"
        print(f"(user reply: {user_input})")
        # self.cur_result_index += 1
    
    def get_rule_prompt(self):
        return RESULTS_PROMPT.format(rubric=self.rubric, item=self.results[self.cur_result_index])
    
    def parse_rule_output(self, output: str):
        # parse the descriptions from <rule>description</rule>
        pattern = r"<rule>\n?(.*?)\n?</rule>"
        matches = re.finditer(pattern, output, re.DOTALL)
        return [str(match.group(1).strip()) for match in matches]
    
    async def execute_get_rule_string(self):
        print("=" * 100)
        prompt = self.get_rule_prompt()
        output = (await get_llm_completions_async(
            [
                [
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ]
            ],
            PROVIDER_PREFERENCES.handle_refinement_message,
            max_new_tokens=8192,
            timeout=180.0,
            use_cache=True,
        ))[0].completions[0].text
        assert output is not None
        rules = self.parse_rule_output(output)
        print(f"Is there a general class of items that {"" if self.item_feedback else "should not "}belong in the rubric? Some suggestions:\n{'\n'.join('-' + rule for rule in rules)}")
        time.sleep(0.5)
        self.rule_feedback = input()
        print(f"(user reply: {self.rule_feedback})")
    
    async def execute_get_cluster(self):
        clusters = await self.assigner.assign(
            self.results,
            [self.rule_feedback,] * len(self.results),
            assignment_callback=None,
        )
        indices = [i for i, c in enumerate(clusters) if c is not None and c[0]]
        print(f"Found {len(indices)} / {len(self.results)} items that belong to the rule. Should we add this rule to the rubric? Reply y/n")
        self.results = [self.results[i] for i in range(len(self.results)) if i not in indices]
        time.sleep(0.5)
        user_input = input()
        self.rule_addition_feedback = user_input == "y"
        print(f"(user reply: {user_input})")
    
    async def execute_add_rule(self):
        if self.rule_addition_feedback:
            self.rubric.inclusion_rules.append(self.rule_feedback)
            print(f"Rubric updated with new rule")
        else:
            print("Rubric was not updated")
    

    async def conversation_loop(self):
        is_first = True
        while True:
            if self.status == ChatStatus.SELECTED_RESULT:
                self.execute_get_result_string()
                self.status = ChatStatus.GOT_YES_NO
                if not is_first:
                    break
            elif self.status == ChatStatus.GOT_YES_NO:
                await self.execute_get_rule_string()
                self.status = ChatStatus.GOT_RULE
            elif self.status == ChatStatus.GOT_RULE:
                await self.execute_get_cluster()
                self.status = ChatStatus.GOT_CLUSTER
            elif self.status == ChatStatus.GOT_CLUSTER:
                await self.execute_add_rule()
                self.status = ChatStatus.SELECTED_RESULT
                is_first = False
            

In [ ]:
full_results: list[str] = []
for result in results:
    if result is not None:
        full_results.extend(result)

chat = RefinementChat(cur_rubric, full_results)
await chat.conversation_loop()